In [1]:
import pandas as pd                                          # tables
import numpy as np                                           # math
from sklearn.pipeline import make_pipeline                   # fill blanks -> model
from sklearn.impute import SimpleImputer                     # fills blanks
from sklearn.linear_model import LinearRegression            # our final model type

EVAL_SEASON = 2024                                           # 2024 now (decisions); switch to 2025 after Sunday's freeze
TARGET = "fantasy_points_ppr"                                # what we predict
ROLE_CUTOFF = 0.25                                           # real-role threshold (Decision 3)
POSITIONS = ["QB", "RB", "WR", "TE"]                         # one model each
ID_COLS = ["player_id", "player_display_name", "position", "team",   # identifiers, never model inputs
           "opponent_team", "season", "week", "history_flag", "is_upcoming"]

df = pd.read_parquet("../data/processed/features.parquet")              # features (v2)
clean = pd.read_parquet("../data/processed/player_weeks_clean.parquet")  # actual game stats (for TDs, snap %)
FEATURES = [c for c in df.columns if c not in ID_COLS + [TARGET]]       # the 26 features

df = df.sort_values(["player_id", "season", "week"])        # each player's games in order
df["games_before"] = df.groupby(["player_id", "season"]).cumcount()  # how many earlier games this season (0 = first)

actual = clean[["player_id", "season", "week",               # what really happened in each game:
                "rushing_tds", "receiving_tds", "passing_tds",   # touchdowns scored
                "offense_pct"]]                              # snap % THIS game (analysis only, never a feature)
df = df.merge(actual, on=["player_id", "season", "week"],    # attach to each row
              how="left", validate="one_to_one")             # one stats row per player-game

df = df[(df["history_flag"] == "ok") & (df["is_upcoming"] == 0)]  # same rows as every other result
train = df[df["season"].between(2020, EVAL_SEASON - 1)]      # learn on every season before the one we analyze
ev = df[df["season"] == EVAL_SEASON].copy()                  # the season we analyze
print(len(FEATURES), "features |", f"train {len(train):,} | eval {len(ev):,}")  # check: 26 features

26 features | train 20,598 | eval 5,275


In [2]:
ev["pred"] = np.nan                                          # empty prediction column
for pos in POSITIONS:                                        # one model per position, same as notebook 05
    tr = train[train["position"] == pos]                     # this position's training rows
    m = ev["position"] == pos                                # which eval rows are this position
    cols = [c for c in FEATURES if tr[c].notna().any()]      # skip all-blank features (EPA/att for non-QBs)
    model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True),  # identical final model
                          LinearRegression())
    model.fit(tr[cols], tr[TARGET])                          # train
    ev.loc[m, "pred"] = model.predict(ev.loc[m, cols])       # predict this position's games

ev["error"] = ev["pred"] - ev[TARGET]                        # signed error: + = predicted too high, - = too low
ev["abs_err"] = ev["error"].abs()                            # points missed by (for MAE)
ev["abs_err_base"] = (ev["fantasy_points_ppr_last3"] - ev[TARGET]).abs()  # Baseline 2's miss, same games
ev["real_role"] = ev["offense_pct_last3"] >= ROLE_CUTOFF    # headline group flag
print("blank predictions:", ev["pred"].isna().sum())         # check: 0

blank predictions: 0


In [3]:
def summarize(data, by):                                     # data = rows to summarize; by = column(s) to group on
    out = data.groupby(by, observed=True).agg(               # one row per group
        games=("abs_err", "size"),                           # how many games in the group
        model_mae=("abs_err", "mean"),                       # model's average miss
        baseline_mae=("abs_err_base", "mean"),               # Baseline 2's average miss, same games
        bias=("error", "mean"),                              # average signed error: - = model guesses too LOW
    )
    out["model_vs_base_pct"] = ((out["baseline_mae"] - out["model_mae"])   # % better than the baseline
                                / out["baseline_mae"] * 100)               # (negative = baseline wins here)
    return out.round(2)                                      # tidy numbers

rr = ev[ev["real_role"]].copy()                              # real-role players (headline group)

In [4]:
rr["history"] = pd.cut(rr["games_before"],                   # bucket by earlier games this season
                       bins=[0, 1, 2, 3, 99],                # (0,1], (1,2], (2,3], (3,99]
                       labels=["1 game", "2 games", "3 games", "4+ games"])  # readable labels
summarize(rr, "history")                                     # error by amount of history

,games,model_mae,baseline_mae,bias,model_vs_base_pct
history,,,,,
1 game,326,4.88,5.44,0.28,10.33
2 games,332,5.21,5.66,-0.43,7.97
3 games,340,4.74,4.81,0.09,1.47
4+ games,3091,5.13,5.51,-0.30,6.87


In [5]:
td = rr["rushing_tds"].fillna(0) + rr["receiving_tds"].fillna(0)        # TDs scored by running or catching
td = td + np.where(rr["position"] == "QB", rr["passing_tds"].fillna(0), 0)  # QBs also get credit for TD passes
rr["tds"] = pd.cut(td, bins=[-1, 0, 1, 2, 99],               # bucket: 0, 1, 2, 3+
                   labels=["0 TD", "1 TD", "2 TD", "3+ TD"])
print(summarize(rr, "tds"))                                  # error by TD count, all positions
rr.groupby(["position", "tds"], observed=True)["abs_err"].mean().unstack().round(2)  # same, split by position

       games  model_mae  baseline_mae   bias  model_vs_base_pct
tds                                                            
0 TD    2815       4.09          4.51   2.47               9.29
1 TD     859       6.05          6.53  -4.63               7.31
2 TD     287       9.26          9.23  -8.81              -0.36
3+ TD    128      11.22         10.83 -11.04              -3.63


tds,0 TD,1 TD,2 TD,3+ TD
position,,,,
QB,8.52,3.85,4.32,9.67
RB,4.62,5.35,12.04,19.02
TE,3.13,6.76,14.24,32.82
WR,4.07,7.15,16.52,32.35


In [6]:
change = rr["offense_pct"] - rr["offense_pct_last3"]         # this game's snap % minus recent average
rr["role_change"] = pd.cut(change, bins=[-9, -0.25, 0.25, 9],  # big drop / normal / big jump (25+ points of snap %)
                           labels=["role shrank", "stable", "role grew"])
summarize(rr, "role_change")                                 # error when a player's role suddenly changes

,games,model_mae,baseline_mae,bias,model_vs_base_pct
role_change,,,,,
role shrank,295,6.09,6.15,5.05,1.08
stable,3596,4.96,5.35,-0.45,7.30
role grew,198,5.91,6.44,-4.19,8.20


In [7]:
summarize(ev, ["position", "real_role"])                     # all players, split by position and role

games  model_mae  baseline_mae  bias  model_vs_base_pct
position real_role                                                         
QB       False         67       4.96          3.74  0.14             -32.74
         True         519       6.27          6.87 -0.55               8.70
RB       False        530       2.98          2.83  0.32              -5.51
         True         858       5.49          5.75  0.15               4.52
TE       False        132       1.94          2.02 -0.66               3.97
         True         964       3.86          4.12 -0.20               6.48
WR       False        457       2.15          1.92 -0.07             -12.31
         True        1748       5.22          5.64 -0.35               7.48

In [8]:
show = ["player_display_name", "position", "team", "opponent_team", "week",   # who and when
        "pred", "fantasy_points_ppr", "fantasy_points_ppr_last3",             # predicted, actual, baseline
        "tds", "offense_pct_last3", "offense_pct"]                             # TDs, recent snap %, this game's snap %
rr.nlargest(15, "abs_err")[show].round(2)                    # 15 largest misses among real-role players

,player_display_name,position,team,opponent_team,week,pred,fantasy_points_ppr,fantasy_points_ppr_last3,tds,offense_pct_last3,offense_pct
21339,Jauan Jennings,WR,SF,LA,3,8.06,46.50,8.55,3+ TD,0.52,0.85
25974,Ja'Marr Chase,WR,CIN,BAL,10,17.19,55.40,16.43,3+ TD,0.90,0.94
15245,Josh Allen,QB,BUF,LA,14,18.94,51.88,23.53,3+ TD,0.91,1.00
8306,Taysom Hill,TE,NO,CLE,11,8.70,41.52,9.74,3+ TD,0.42,0.55
23197,Jerry Jeudy,WR,CLE,DEN,13,10.13,40.50,18.33,1 TD,0.89,0.91
34818,Zach Charbonnet,RB,SEA,ARI,14,8.13,38.30,5.60,2 TD,0.38,0.80
25977,Ja'Marr Chase,WR,CIN,DAL,14,15.47,45.10,34.17,2 TD,0.97,0.93
3495,Davante Adams,WR,NYJ,JAX,15,13.73,42.80,18.90,2 TD,0.92,0.93
34535,Jahmyr Gibbs,RB,DET,MIN,18,17.20,46.00,26.70,3+ TD,0.65,0.66
10723,Alvin Kamara,RB,NO,DAL,2,15.30,44.00,22.00,3+ TD,0.52,0.63


In [9]:
tag = f"_{EVAL_SEASON}"                                      # file name suffix, e.g. _2024
summarize(rr, "history").to_csv(f"../reports/tables/errors_by_history{tag}.csv")     # early weeks
summarize(rr, "tds").to_csv(f"../reports/tables/errors_by_tds{tag}.csv")             # touchdown games
summarize(rr, "role_change").to_csv(f"../reports/tables/errors_by_role_change{tag}.csv")  # role changes
summarize(ev, ["position", "real_role"]).to_csv(f"../reports/tables/errors_by_player_type{tag}.csv")  # player type
rr.nlargest(15, "abs_err")[show].to_csv(f"../reports/tables/biggest_misses{tag}.csv", index=False)  # top misses
print("Saved error tables with suffix", tag)                 # confirm

Saved error tables with suffix _2024
